# 02 · Core data, core–log depth matching, density check and borehole-quality flags
**F-O gas field, Bredasdorp Basin — Notebook 02**

Inputs:
* the eight `*_composite_v2.csv` files from **F-O_composites_v2.zip** (one folder);
* `core_data_raw_unshifted.csv` (850 plugs, 6 wells, read from the scanned SOEKOR / Corex / TerraTek reports — the OCR text is in
  `core_ocr_text/` so every value can be checked against the scan).

Outputs (to `OUT_DIR`): per-core depth shifts, shifted core data, density-vs-core bias table, composite **v3** LAS/CSV per well
(adds BS, CAL_BS, BADHOLE, despiked sonic, core curves CPOR/CKL/CRHOG, F-S1 RHOB_RAW; removes F-O1 RHOB_ALT) and a core–log
figure per cored well.

Conventions: depth in m below KB; **log depth = core depth + shift**.

In [ ]:
# Cell 1 — paths: EDIT these three lines
from pathlib import Path
COMP_DIR = Path(r"C:\Users\YOUR_NAME\...\F-O_composites_v2")        # unzipped F-O_composites_v2 (CSV + LAS files)
CORE_CSV = Path(r"C:\Users\YOUR_NAME\...\FO_petro_v3\core_data_raw_unshifted.csv")
OUT_DIR  = Path(r"C:\Users\YOUR_NAME\...\F-O_v3_outputs")
OUT_DIR.mkdir(exist_ok=True, parents=True)

In [ ]:
# Cell 2 — load
import numpy as np, pandas as pd, matplotlib.pyplot as plt, lasio, warnings
warnings.filterwarnings("ignore")
from petrotools import coretools as ct
WELLS = ["F-O1", "F-O2", "F-O3", "F-O4", "F-O6", "F-O8", "F-R1", "F-S1"]
logs = {w: pd.read_csv(COMP_DIR / f"{w}_composite_v2.csv", index_col=0) for w in WELLS}
core = pd.read_csv(CORE_CSV)
core.groupby("well").agg(plugs=("depth_m", "size"), cores=("core", "nunique"), top=("depth_m", "min"),
                         base=("depth_m", "max"), phi_median=("phi_pct", "median"), k_median_mD=("kkl_md", "median")).round(2)

## Step 1 — core-to-log depth shift, core by core
Each core was cut on drill-pipe depth; the logs are on wireline depth. For every core we slide the plug porosities along the
density log (−8 to +8 m, 0.076 m steps) and keep the shift with the highest correlation between core porosity and −RHOB.
A match is **robust** when r ≥ 0.75, it beats any peak more than 1.5 m away by ≥ 0.10, and the core has ≥ 6 plugs.

In [ ]:
# Cell 3 — scan every core
rows, scans = [], {}
for (w, k), cc in core.groupby(["well", "core"]):
    if cc.phi_pct.notna().sum() < 6:
        rows.append(dict(well=w, core=k, n=int(cc.phi_pct.notna().sum()), best_shift=np.nan, r_best=np.nan, r_zero=np.nan, second_r=np.nan)); continue
    sc = ct.shift_scan(cc, logs[w], "RHOB"); scans[(w, k)] = sc
    rows.append(dict(well=w, core=k, n=int(cc.phi_pct.notna().sum()), **ct.best_shift(sc)))
scan_tbl = pd.DataFrame(rows).round(3)
scan_tbl

In [ ]:
# Cell 4 — the correlation-vs-shift curves (a sharp single peak = a trustworthy shift)
fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharey=True)
for ax, w in zip(axes.flat, ["F-O1", "F-O2", "F-O3", "F-O4", "F-O6", "F-S1"]):
    for (ww, k), sc in scans.items():
        if ww == w: ax.plot(sc.shift_m, sc.r, lw=1, label=f"core {k}")
    ax.axvline(0, color="0.6", lw=0.8); ax.set_title(w, fontsize=9); ax.legend(fontsize=7); ax.grid(alpha=0.3)
for ax in axes[-1]: ax.set_xlabel("shift (m)  [log = core + shift]")
for ax in axes[:, 0]: ax.set_ylabel("r(core φ, −RHOB)")
plt.tight_layout(); plt.savefig(OUT_DIR / "core_shift_scans.png", dpi=150)

**F-O4 needs a multi-log check.** Its 80 m of core (3728–3809 m) is three core runs in a very uniform sand (φ 8–11 %), so a
single density match is weak. Below, the same scan is run on RHOB, DT and DTCO per core; where they agree, their consensus is used
(override table in Cell 6).

In [ ]:
# Cell 5 — F-O4: density + sonic per core
chk = []
for k, cc in core[core.well == "F-O4"].groupby("core"):
    for cv, sign in [("RHOB", -1), ("DT", 1), ("DTCO", 1)]:
        chk.append(dict(core=k, curve=cv, **ct.best_shift(ct.shift_scan(cc, logs["F-O4"], cv, sign=sign))))
pd.DataFrame(chk).round(2)

In [ ]:
# Cell 6 — decide the shift for every core
OVERRIDES = {
    ("F-O4", 1): (2.95, "multi-log consensus: RHOB +3.05, DTCO +2.90, DT +2.82"),
    ("F-O4", 3): (2.70, "multi-log consensus: RHOB +2.82, DTCO +2.52, DT +2.52"),
    ("F-O4", 2): (2.83, "mean of F-O4 cores 1 and 3 (core 2 too homogeneous: r < 0.23 on all logs)"),
}
shifts = ct.choose_shifts(scan_tbl, OVERRIDES)
rng = core.groupby(["well", "core"]).depth_m.agg(core_top_m="min", core_base_m="max", plugs="size").reset_index()
shifts = rng.merge(shifts, on=["well", "core"])
PUBLISHED = {"F-O1": "RPS 2008 +3.5; SOEKOR +2", "F-O2": "RPS 2008 +4.5; SOEKOR cores 1-4 +1, core 5 +4.5",
             "F-O3": "RPS 2008 +2.75", "F-O4": "RPS 2008 +2.75"}
shifts["published"] = shifts.well.map(PUBLISHED).fillna("")
shifts.to_csv(OUT_DIR / "core_log_depth_shifts.csv", index=False)
pd.set_option("display.max_colwidth", 90); shifts

**Reading the result.** F-O2's five cores all need +4.5 to +5.0 m — the RPS (2008) value, not SOEKOR's +1 m for cores 1–4.
F-O3 (+2.4 to +2.7 m) and F-O4 (+2.7 to +3.0 m) agree with RPS (+2.75 m). F-O1 comes out at +2.9 m (moderate confidence:
one core), between the published +2 m and +3.5 m. F-O6 (+2.0 m) and F-S1 (+2.2 to +2.4 m) have no published shift.
F-S1 core 1 does not match on any log and takes the F-S1 median — flag it in the paper.

## Step 2 — apply shifts and test the density log against core porosity

In [ ]:
# Cell 7
core_s = ct.apply_shifts(core, shifts[["well", "core", "shift_m"]])
core_s.to_csv(OUT_DIR / "core_data_shifted.csv", index=False)
bias = []
for w, g in core_s.groupby("well"):
    for cv in [c for c in ("RHOB", "RHOB_ALT") if c in logs[w]]:
        bias.append(dict(well=w, curve=cv, **ct.density_bias(g, logs[w], cv)))
    if "DRHO" in logs[w]:
        bias.append(dict(well=w, curve="RHOB-DRHO", **ct.density_bias(g, logs[w], logs[w].RHOB - logs[w].DRHO)))
bias = pd.DataFrame(bias); bias.to_csv(OUT_DIR / "core_density_bias.csv", index=False); bias

**Decisions from this table** (ρma = plug grain density, ρf = 1.0 g/cc):
* **F-O1: use RHOB, drop RHOB_ALT.** RHOB matches core with ≈ 0 pu bias; RHOB_ALT (the 12 Sep field copy) is 5 pu too porous.
* **F-S1: use RHOB_RAW = RHOB − DRHO.** The corrected RHOB reads ~6 pu too tight; without the DRHO correction the bias is
  −0.4 pu and the correlation improves — the compensation over-corrected (barite mud, PEF ≈ 4).
* **F-O3: −3 pu offset that DRHO does not explain** (≈ +0.05 g/cc density offset). Carry it as a per-well calibration term in
  the porosity model (Notebook 03) rather than editing the log.
* F-O2, F-O4, F-O6 are within ~1 pu: no change.

## Step 3 — bit size, bad-hole flag, sonic despike, core curves → composite v3

In [ ]:
# Cell 8
notes_all = []
for w in WELLS:
    d, notes = ct.postprocess(w, logs[w])
    if "RHOB_ALT" in d: d = d.drop(columns="RHOB_ALT"); notes["RHOB_ALT"] = "removed (core: +5 pu bias)"
    if w == "F-S1": d["RHOB_RAW"] = d.RHOB - d.DRHO; notes["RHOB_RAW"] = "RHOB - DRHO (matches core; RHOB 6 pu low)"
    d = d.join(ct.core_curves(core_s[core_s.well == w], d.index.to_numpy()))
    d.to_csv(OUT_DIR / f"{w}_composite_v3.csv", float_format="%.5f")
    las = lasio.LASFile(); las.well["WELL"].value = w; las.well["FLD"].value = "F-O Gas Field"
    las.other = f"{w} composite v3 (Notebook 02). " + " | ".join(f"{k}: {v}" for k, v in notes.items())
    units = {c.mnemonic: c.unit for c in lasio.read(str(COMP_DIR / f"{w}_composite_v2.las")).curves}
    units.update(BS="in", CAL_BS="in", BADHOLE="", CPOR="V/V", CKL="mD", CRHOG="g/cm3", RHOB_RAW="g/cm3")
    las.append_curve("DEPT", d.index.to_numpy(), unit="M")
    for c in d.columns: las.append_curve(c, d[c].to_numpy(), unit=units.get(c, ""))
    las.write(str(OUT_DIR / f"{w}_composite_v3.las"), version=2.0, fmt="%.5f")
    r = d.loc[3830:4040] if w == "F-R1" else d.loc[3590:3950]
    notes_all.append(dict(well=w, bit_size=notes["BS"], reservoir_badhole_pct=round(100 * r.BADHOLE.mean(), 1),
                          core_points=int(d.get("CPOR", pd.Series(dtype=float)).notna().sum())))
pd.DataFrame(notes_all)

**About the flag.** `BADHOLE = 1` where the caliper exceeds bit size by > 3 in **or** |DRHO| > 0.10 g/cc. A milder threshold
(1.5 in) flags 50–70 % of F-O1/F-O4/F-R1 although their DRHO stays < 0.05 — the hole is oval/slightly enlarged but the pad still
reads the formation. `CAL_BS` is written so you can apply your own threshold in IP.

## Step 4 — core–log figures (one per cored well)

In [ ]:
# Cell 9
def core_fig(w):
    d = pd.read_csv(OUT_DIR / f"{w}_composite_v3.csv", index_col=0); c = core_s[core_s.well == w]
    s = d.loc[c.depth_m.min() - 8: c.depth_log_m.max() + 8]; rb = "RHOB_RAW" if "RHOB_RAW" in d else "RHOB"
    phid = (100 * (2.67 - s[rb]) / 1.67).mask(s.BADHOLE == 1)
    f, ax = plt.subplots(1, 3, figsize=(7.2, 9), sharey=True, gridspec_kw=dict(width_ratios=[1, 1.6, 1.4]))
    ax[0].plot(s.GR, s.index, lw=1); ax[0].set_xlim(0, 150); ax[0].set_title("GR (API)", fontsize=9)
    ax[1].plot(phid, s.index, lw=1, label=f"φD from {rb}")
    ax[1].scatter(c.phi_pct, c.depth_m, s=10, facecolors="none", edgecolors="0.5", lw=0.8, label="core, core depth")
    ax[1].scatter(c.phi_pct, c.depth_log_m, s=12, color="#eb6834", zorder=3, label="core, log depth")
    ax[1].set_xlim(25, 0); ax[1].set_title("Porosity (%)", fontsize=9); ax[1].legend(fontsize=7, loc="lower left")
    ax[2].scatter(c.kkl_md, c.depth_log_m, s=12, color="#eb6834", label="core Kkl")
    if "KTIM" in s: ax[2].plot(s.KTIM.clip(lower=1e-4), s.index, lw=1, label="NMR KTIM")
    ax[2].set_xscale("log"); ax[2].set_xlim(1e-3, 1e3); ax[2].set_title("Permeability (mD)", fontsize=9); ax[2].legend(fontsize=7, loc="lower left")
    for a in ax: a.grid(alpha=0.3)
    ax[0].invert_yaxis(); ax[0].set_ylabel("Logger depth (m below KB)")
    t = shifts[shifts.well == w]; f.suptitle(f"{w} · " + "; ".join(f"core {k}: {v:+.2f} m" for k, v in zip(t.core, t.shift_m)), fontsize=9)
    f.tight_layout(rect=(0, 0, 1, 0.96)); f.savefig(OUT_DIR / f"{w}_core_depth_match.png", dpi=150); plt.close(f)
for w in ["F-O1", "F-O2", "F-O3", "F-O4", "F-O6", "F-S1"]: core_fig(w)
from IPython.display import Image; Image(filename=str(OUT_DIR / "F-O2_core_depth_match.png"))

## What this notebook establishes (for the methods section)
1. **850 routine core plugs from 20 cores in 6 wells**, digitised from the original reports, with every value traceable to the
   OCR text and the scan.
2. **Core-to-log depth shifts per core** from correlation with the density log (r = 0.73–0.99 where robust), resolving the
   RPS (2008) vs SOEKOR (1993) disagreement in favour of RPS for F-O2.
3. **Density calibration against core:** F-O1 RHOB (not RHOB_ALT), F-S1 RHOB − DRHO, F-O3 needs a calibration term.
4. **Borehole-quality flags** (BS, CAL_BS, BADHOLE) and despiked sonic, ready for the IP deterministic and mineral-solver work.

**Next (Notebook 03):** shale volume and porosity models calibrated to these core points (per-well density calibration,
gas correction), then Rw / m / n from the SCAL (F-O2, F-O6) for saturation.